# Kiwi 명사/명사구 기반 seed 사전 확장 실험

목적:

- `final/00_corpus.ipynb`와 같은 방식으로 DART 사업보고서의 II/IV/VI 섹션을 기업-연도 문서로 만든다.
- Kiwi 형태소 분석기로 코퍼스에서 실제 등장한 명사와 명사구 후보만 추출한다.
- `data/seed_dictionary.csv`의 E/S/G seed와 후보 단어를 embedding vector로 바꾼 뒤 cosine similarity를 계산한다.
- 유사도 threshold별로 남는 후보 개수와 후보 단어 목록을 확인한다.

주의:

- ESG 등급은 이 노트북의 후보 선택에 사용하지 않는다.
- 이 노트북의 목적은 성능 튜닝이 아니라, seed 기반 확장 후보의 coverage/noise를 진단하는 것이다.

In [1]:
from pathlib import Path
import html
import math
import re
import subprocess
import sys
import unicodedata
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from IPython.display import display

try:
    from kiwipiepy import Kiwi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kiwipiepy"])
    from kiwipiepy import Kiwi

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

## 1. 경로와 실험 설정

In [2]:
if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")

ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/My Drive/UD_26"),
    Path.cwd(),
    Path.cwd().parent,
]


def first_existing(candidates, label):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"No existing path for {label}: {candidates}")


ROOT = first_existing([p for p in ROOT_CANDIDATES if (p / "data").exists() and (p / "final").exists()], "ROOT")
DATA_DIR = ROOT / "data"
FINAL_DIR = ROOT / "final"
RAW_XML_DIR = first_existing([
    FINAL_DIR / "raw_xml",
    DATA_DIR / "dart" / "raw_xml",
    ROOT / "raw_xml",
], "RAW_XML_DIR")
SEED_DICTIONARY_PATH = first_existing([
    DATA_DIR / "seed_dictionary.csv",
    FINAL_DIR / "seed_dictionary.csv",
    ROOT / "seed_dictionary.csv",
], "SEED_DICTIONARY_PATH")
COMPANY_MASTER_PATH = first_existing([
    DATA_DIR / "company_master.csv",
    FINAL_DIR / "company_master.csv",
    ROOT / "company_master.csv",
], "COMPANY_MASTER_PATH")
OUTPUT_DIR = FINAL_DIR / "kiwi_embedding_seed_expansion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 속도 점검용으로 먼저 작게 돌릴 때는 정수로 바꾸세요. 전체 실행은 None.
MAX_FILES = None

# 후보 단어 풀 필터
MIN_TERM_FREQ = 3
MIN_DOC_FREQ = 2
MAX_CANDIDATES = 30000

# embedding 모델. 한국어 공시 문맥을 반영한 후보 랭킹용으로 BGE-M3-ko를 기본값으로 둔다.
MODEL_NAME = "dragonkue/BGE-m3-ko"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64 if DEVICE == "cuda" else 16

THRESHOLDS = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
TOP_N_PER_SEED = 100

print("ROOT:", ROOT)
print("RAW_XML_DIR:", RAW_XML_DIR)
print("SEED_DICTIONARY_PATH:", SEED_DICTIONARY_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_NAME:", MODEL_NAME)
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("CUDA device:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT: /content/drive/MyDrive/UD_26
RAW_XML_DIR: /content/drive/MyDrive/UD_26/final/raw_xml
SEED_DICTIONARY_PATH: /content/drive/MyDrive/UD_26/final/seed_dictionary.csv
OUTPUT_DIR: /content/drive/MyDrive/UD_26/final/kiwi_embedding_seed_expansion
MODEL_NAME: dragonkue/BGE-m3-ko
DEVICE: cuda
CUDA device: Tesla T4


## 2. `00_corpus.ipynb` 방식으로 DART II/IV/VI 섹션 코퍼스 만들기

In [ ]:
TARGET_TITLE_REGEX = {
    "II. 사업의 내용": r"^(II|Ⅱ)\.\s*사업의\s*내용",
    "IV. 이사의 경영진단 및 분석의견": r"^(IV|Ⅳ)\.\s*이사의\s*경영진단\s*및\s*분석의견",
    "VI. 이사회 등 회사의 기관에 관한 사항": r"^(VI|Ⅵ)\.\s*이사회\s*등\s*회사의\s*기관에\s*관한\s*사항",
}

TITLE_RE = re.compile(r"<TITLE\b[^>]*>(.*?)</TITLE>", flags=re.I | re.S)
MAIN_TITLE_RE = re.compile(
    r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X|Ⅰ|Ⅱ|Ⅲ|Ⅳ|Ⅴ|Ⅵ|Ⅶ|Ⅷ|Ⅸ|Ⅹ)\."
)


def clean_xml_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_report_text(text: str) -> str:
    text = clean_xml_text(text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_xml_filename(path: Path) -> tuple[str, int, str]:
    match = re.match(r"(\d{6})_(\d{4})_(\d+)\.xml$", path.name)
    if not match:
        raise ValueError(f"Unexpected XML filename: {path.name}")
    stock_code, fiscal_year, rcept_no = match.groups()
    return stock_code, int(fiscal_year), rcept_no


def extract_target_sections(xml_text: str) -> list[dict]:
    titles = []
    for match in TITLE_RE.finditer(xml_text):
        title = clean_xml_text(match.group(1))
        if MAIN_TITLE_RE.match(title):
            titles.append((title, match.start()))

    sections = []
    for i, (title, start) in enumerate(titles):
        section_name = None
        for name, pattern in TARGET_TITLE_REGEX.items():
            if re.search(pattern, title):
                section_name = name
                break
        if section_name is None:
            continue
        end = titles[i + 1][1] if i + 1 < len(titles) else len(xml_text)
        section_raw = xml_text[start:end]
        sections.append({"section": section_name, "text": clean_xml_text(section_raw)})
    return sections


def load_company_names(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=["stock_code", "company_name"])
    company_df = pd.read_csv(path, dtype={"stock_code": "string"})
    company_df["stock_code"] = (
        company_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
    )
    if "company_name" not in company_df.columns:
        return pd.DataFrame(columns=["stock_code", "company_name"])
    return company_df[["stock_code", "company_name"]].dropna().drop_duplicates("stock_code")


def build_firm_year_corpus(raw_xml_dir: Path, max_files: int | None = None) -> pd.DataFrame:
    rows = []
    xml_files = sorted(raw_xml_dir.glob("*.xml"))
    if max_files is not None:
        xml_files = xml_files[:max_files]
    if not xml_files:
        raise FileNotFoundError(f"No XML files found in {raw_xml_dir}")

    for xml_path in xml_files:
        stock_code, fiscal_year, rcept_no = parse_xml_filename(xml_path)
        xml_text = xml_path.read_text(encoding="utf-8", errors="ignore")
        sections = extract_target_sections(xml_text)
        document = " ".join(section["text"] for section in sections)
        document_norm = normalize_report_text(document)
        rows.append({
            "stock_code": stock_code,
            "fiscal_year": fiscal_year,
            "rcept_no": rcept_no,
            "file_name": xml_path.name,
            "document": document,
            "document_norm": document_norm,
            "section_count": len({section["section"] for section in sections}),
            "total_word_count": len(document_norm.split()),
            "total_char_count": len(document_norm),
            "esg_year": fiscal_year + 1,
        })

    corpus_df = pd.DataFrame(rows)
    company_df = load_company_names(COMPANY_MASTER_PATH)
    corpus_df = corpus_df.merge(company_df, on="stock_code", how="left")
    ordered_cols = [
        "stock_code", "company_name", "fiscal_year", "rcept_no", "file_name",
        "document", "document_norm", "section_count", "total_word_count",
        "total_char_count", "esg_year",
    ]
    return corpus_df[ordered_cols].sort_values(["stock_code", "fiscal_year", "rcept_no"]).reset_index(drop=True)


corpus_df = build_firm_year_corpus(RAW_XML_DIR, MAX_FILES)
print("rows:", len(corpus_df))
print("section_count distribution:")
print(corpus_df["section_count"].value_counts().sort_index())
display(corpus_df[["stock_code", "company_name", "fiscal_year", "section_count", "total_word_count"]].head())

rows: 381
section_count distribution:
section_count
3    381
Name: count, dtype: int64


,stock_code,company_name,fiscal_year,section_count,total_word_count
0,000020,동화약품,2022,3,7863
1,000020,동화약품,2023,3,8120
2,000020,동화약품,2024,3,8152
3,000040,KR모터스,2022,3,4256
4,000040,KR모터스,2023,3,4943


## 3. seed 사전 로드와 seed query 만들기

In [4]:
def normalize_text(value) -> str:
    text = "" if pd.isna(value) else str(value)
    text = unicodedata.normalize("NFKC", html.unescape(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_term(value) -> str:
    text = normalize_text(value)
    return text.strip(" \t\r\n\"'`.,;:()[]{}<>")


def split_seed_terms(row: pd.Series) -> list[str]:
    values = [row.get("seed_term", "")]
    pattern = row.get("pattern", "")
    if pd.notna(pattern):
        values.extend(str(pattern).split("|"))

    terms = []
    seen = set()
    for value in values:
        term = normalize_term(value)
        if not term or term.lower() == "nan" or term in seen:
            continue
        seen.add(term)
        terms.append(term)
    return terms


seed_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")
required_cols = {"dimension", "seed_term", "pattern"}
missing = required_cols - set(seed_df.columns)
if missing:
    raise ValueError(f"seed_dictionary.csv missing columns: {sorted(missing)}")

seed_rows = []
for seed_idx, row in seed_df.reset_index(drop=True).iterrows():
    dimension = normalize_term(row["dimension"])
    if dimension not in {"E", "S", "G"}:
        continue
    seed_term = normalize_term(row["seed_term"])
    for query_term in split_seed_terms(row):
        seed_rows.append({
            "seed_id": f"{dimension}_{seed_idx:03d}",
            "dimension": dimension,
            "seed_term": seed_term,
            "query_term": query_term,
            # 단어 하나만 넣는 것보다 seed의 문헌/메모 맥락을 짧게 붙여 semantic anchor를 안정화한다.
            "query_text": " ".join([
                query_term,
                normalize_text(row.get("source_basis", "")),
                normalize_text(row.get("notes", "")),
            ]).strip(),
        })

seed_query_df = pd.DataFrame(seed_rows).drop_duplicates(["dimension", "seed_term", "query_term"])
seed_terms = set(seed_query_df["query_term"])

print("seed rows:", len(seed_df))
print("seed query terms:", len(seed_query_df))
display(seed_query_df.groupby("dimension").size().rename("seed_query_count").reset_index())
display(seed_query_df.head(20))

seed rows: 30
seed query terms: 54


,dimension,seed_query_count
0,E,18
1,G,18
2,S,18


,seed_id,dimension,seed_term,query_term,query_text
0,E_000,E,탄소,탄소,탄소 Bloomberg ESG climate change; Sautner-style climate exposure carbon / climate exposure core seed
1,E_001,E,온실가스,온실가스,온실가스 Bloomberg ESG climate change; GHG emissions literature greenhouse gas emissions seed
2,E_001,E,온실가스,GHG,GHG Bloomberg ESG climate change; GHG emissions literature greenhouse gas emissions seed
3,E_002,E,탄소중립,탄소중립,탄소중립 Bloomberg ESG climate change; net-zero transition literature net-zero transition Korean term
4,E_003,E,넷제로,넷제로,넷제로 Bloomberg ESG climate change; net-zero transition literature net-zero borrowed term
5,E_003,E,넷제로,net zero,net zero Bloomberg ESG climate change; net-zero transition literature net-zero borrowed term
6,E_003,E,넷제로,net-zero,net-zero Bloomberg ESG climate change; net-zero transition literature net-zero borrowed term
7,E_004,E,재생에너지,재생에너지,재생에너지 Bloomberg ESG water/energy management; climate opportunity literature renewable energy seed
8,E_004,E,재생에너지,renewable energy,renewable energy Bloomberg ESG water/energy management; climate opportunity literature renewable energy seed
9,E_005,E,에너지,에너지,에너지 Bloomberg ESG water/energy management energy management broad seed


## 4. Kiwi로 명사와 명사구 후보 추출

In [5]:
kiwi = Kiwi()

NOUN_TAGS = {"NNG", "NNP", "SL"}


def is_good_term(term: str) -> bool:
    term = normalize_term(term)
    if len(term) < 2:
        return False
    if term.isdigit():
        return False
    if re.fullmatch(r"[0-9.,%/년월일기제]+", term):
        return False
    if re.search(r"[\u3131-\u318E]", term):
        return False
    return True


def kiwi_noun_candidates(text: str) -> list[str]:
    tokens = kiwi.tokenize(text)
    candidates = []
    current = []

    for token in tokens:
        form = normalize_term(token.form)
        tag = token.tag

        if tag in NOUN_TAGS and is_good_term(form):
            candidates.append(form)
            current.append(form)
        else:
            if len(current) >= 2:
                # 긴 연속 명사구 전체와 인접 bigram을 같이 후보로 남긴다.
                candidates.append(" ".join(current))
                for i in range(len(current) - 1):
                    candidates.append(" ".join(current[i:i + 2]))
            current = []

    if len(current) >= 2:
        candidates.append(" ".join(current))
        for i in range(len(current) - 1):
            candidates.append(" ".join(current[i:i + 2]))

    return [term for term in candidates if is_good_term(term)]


term_counter = Counter()
doc_counter = Counter()
kiwi_terms_by_doc = []

for text in corpus_df["document_norm"]:
    terms = kiwi_noun_candidates(text)
    kiwi_terms_by_doc.append(terms)
    term_counter.update(terms)
    doc_counter.update(set(terms))

candidate_rows = []
for term, freq in term_counter.most_common():
    if term in seed_terms:
        continue
    doc_freq = doc_counter[term]
    if freq < MIN_TERM_FREQ or doc_freq < MIN_DOC_FREQ:
        continue
    candidate_rows.append({
        "candidate_term": term,
        "term_frequency": freq,
        "doc_frequency": doc_freq,
    })

candidate_df = pd.DataFrame(candidate_rows)
candidate_df = candidate_df.head(MAX_CANDIDATES).reset_index(drop=True)

print("candidate terms:", len(candidate_df))
display(candidate_df.head(30))
display(candidate_df[["term_frequency", "doc_frequency"]].describe())

candidate terms: 30000


,candidate_term,term_frequency,doc_frequency
0,찬성,81083,365
1,찬성 찬성,61577,346
2,사업,41360,381
3,이사,38865,381
4,자산,36471,381
5,회사,33717,381
6,사항,29949,381
7,시장,28884,381
8,금융,28652,381
9,관리,28546,381


,term_frequency,doc_frequency
count,30000.000000,30000.000000
mean,223.139300,37.860267
std,1285.217896,64.670559
min,18.000000,2.000000
25%,25.000000,9.000000
50%,40.000000,16.000000
75%,91.000000,33.000000
max,81083.000000,381.000000


## 5. Embedding cosine similarity 계산

In [6]:
def encode_texts(model, texts: list[str], batch_size: int) -> np.ndarray:
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return np.asarray(embeddings, dtype=np.float32)


if candidate_df.empty:
    raise ValueError("No candidates left. Lower MIN_TERM_FREQ/MIN_DOC_FREQ or inspect Kiwi extraction.")

model = SentenceTransformer(MODEL_NAME, device=DEVICE)

seed_embeddings = encode_texts(model, seed_query_df["query_text"].tolist(), BATCH_SIZE)
candidate_embeddings = encode_texts(model, candidate_df["candidate_term"].tolist(), BATCH_SIZE)

similarity_matrix = seed_embeddings @ candidate_embeddings.T
print("similarity_matrix:", similarity_matrix.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/31.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

similarity_matrix: (54, 30000)


## 6. 후보별 최고 유사 seed와 E/S/G 차원 판정

In [7]:
best_rows = []

for cand_i, cand in candidate_df.reset_index(drop=True).iterrows():
    sims = similarity_matrix[:, cand_i]
    best_i = int(np.argmax(sims))
    best_seed = seed_query_df.iloc[best_i]

    dim_scores = {}
    for dimension, idx in seed_query_df.groupby("dimension").groups.items():
        dim_scores[dimension] = float(np.max(sims[list(idx)]))

    sorted_dim_scores = sorted(dim_scores.items(), key=lambda x: x[1], reverse=True)
    best_dim, best_dim_score = sorted_dim_scores[0]
    second_dim_score = sorted_dim_scores[1][1] if len(sorted_dim_scores) > 1 else np.nan

    best_rows.append({
        "candidate_term": cand["candidate_term"],
        "term_frequency": cand["term_frequency"],
        "doc_frequency": cand["doc_frequency"],
        "best_dimension": best_dim,
        "best_dimension_score": best_dim_score,
        "second_dimension_score": second_dim_score,
        "dimension_margin": best_dim_score - second_dim_score,
        "best_seed_id": best_seed["seed_id"],
        "best_seed_term": best_seed["seed_term"],
        "best_query_term": best_seed["query_term"],
        "best_query_similarity": float(sims[best_i]),
        "E_score": dim_scores.get("E", np.nan),
        "S_score": dim_scores.get("S", np.nan),
        "G_score": dim_scores.get("G", np.nan),
    })

scored_candidate_df = pd.DataFrame(best_rows)
scored_candidate_df = scored_candidate_df.sort_values(
    ["best_dimension", "best_dimension_score", "dimension_margin", "candidate_term"],
    ascending=[True, False, False, True],
).reset_index(drop=True)

raw_path = OUTPUT_DIR / "kiwi_embedding_raw_candidates.csv"
scored_candidate_df.to_csv(raw_path, index=False, encoding="utf-8-sig")

print("saved:", raw_path)
display(scored_candidate_df.head(30))

saved: /content/drive/MyDrive/UD_26/final/kiwi_embedding_seed_expansion/kiwi_embedding_raw_candidates.csv


,candidate_term,term_frequency,doc_frequency,best_dimension,best_dimension_score,second_dimension_score,dimension_margin,best_seed_id,best_seed_term,best_query_term,best_query_similarity,E_score,S_score,G_score
0,전력 사용량,26,10,E,0.635120,0.295631,0.339489,E_006,전력,전력사용량,0.635120,0.635120,0.284788,0.295631
1,에너지 사용량,382,62,E,0.627394,0.305975,0.321419,E_006,전력,전력사용량,0.627394,0.627394,0.299493,0.305975
2,에너지 사용,104,38,E,0.599126,0.312551,0.286576,E_006,전력,전력사용량,0.599126,0.599126,0.306756,0.312551
3,전력 소비량,21,7,E,0.592073,0.257622,0.334451,E_006,전력,전력사용량,0.592073,0.592073,0.250606,0.257622
4,수질 관리,29,15,E,0.590509,0.268300,0.322209,E_009,폐수,수질,0.590509,0.590509,0.268300,0.262817
5,ESG 관리,19,11,E,0.589035,0.481609,0.107426,E_009,폐수,물관리,0.589035,0.589035,0.405378,0.481609
6,전기 사용량,18,16,E,0.586301,0.293544,0.292757,E_006,전력,전력사용량,0.586301,0.586301,0.293544,0.265268
7,에너지 이용,21,12,E,0.578137,0.321549,0.256587,E_006,전력,전력사용량,0.578137,0.578137,0.318232,0.321549
8,재생 에너지,1139,123,E,0.577841,0.333035,0.244806,E_004,재생에너지,재생에너지,0.577841,0.577841,0.333035,0.280825
9,탄소 중립,759,121,E,0.577353,0.399337,0.178016,E_002,탄소중립,탄소중립,0.577353,0.577353,0.244101,0.399337


## 7. 유사도 threshold별 남는 후보 개수

In [8]:
summary_rows = []

for theta in THRESHOLDS:
    kept = scored_candidate_df[scored_candidate_df["best_dimension_score"] >= theta]
    for dimension in ["E", "S", "G"]:
        sub = kept[kept["best_dimension"] == dimension]
        summary_rows.append({
            "threshold": theta,
            "dimension": dimension,
            "candidate_count": len(sub),
            "median_similarity": sub["best_dimension_score"].median() if len(sub) else np.nan,
            "median_margin": sub["dimension_margin"].median() if len(sub) else np.nan,
            "median_doc_frequency": sub["doc_frequency"].median() if len(sub) else np.nan,
        })

threshold_summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "kiwi_embedding_threshold_summary.csv"
threshold_summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("saved:", summary_path)
display(threshold_summary_df.pivot(index="threshold", columns="dimension", values="candidate_count"))
display(threshold_summary_df)

saved: /content/drive/MyDrive/UD_26/final/kiwi_embedding_seed_expansion/kiwi_embedding_threshold_summary.csv


dimension,E,G,S
threshold,,,
0.45,186,246,36
0.50,75,98,3
0.55,24,26,0
0.60,2,1,0
0.65,0,1,0
0.70,0,1,0
0.75,0,0,0
0.80,0,0,0


,threshold,dimension,candidate_count,median_similarity,median_margin,median_doc_frequency
0,0.45,E,186,0.488624,0.169909,17.5
1,0.45,S,36,0.470744,0.131639,15.0
2,0.45,G,246,0.486775,0.185321,22.0
3,0.50,E,75,0.533647,0.195279,18.0
4,0.50,S,3,0.512178,0.049451,13.0
5,0.50,G,98,0.520928,0.226570,23.0
6,0.55,E,24,0.562508,0.212079,19.0
7,0.55,S,0,NaN,NaN,NaN
8,0.55,G,26,0.568978,0.269026,24.0
9,0.60,E,2,0.631257,0.330454,36.0


## 8. threshold별 남는 단어 확인

In [9]:
def show_terms_by_threshold(theta: float, n: int = 80, min_margin: float | None = None):
    kept = scored_candidate_df[scored_candidate_df["best_dimension_score"] >= theta].copy()
    if min_margin is not None:
        kept = kept[kept["dimension_margin"] >= min_margin]
    kept = kept.sort_values(
        ["best_dimension", "best_dimension_score", "dimension_margin", "doc_frequency"],
        ascending=[True, False, False, False],
    )
    print(f"threshold={theta}, min_margin={min_margin}, kept={len(kept)}")
    for dimension in ["E", "S", "G"]:
        sub = kept[kept["best_dimension"] == dimension].head(n)
        print(f"\n[{dimension}] {len(kept[kept['best_dimension'] == dimension])} candidates")
        display(sub[[
            "candidate_term", "best_dimension_score", "dimension_margin",
            "best_seed_term", "best_query_term", "term_frequency", "doc_frequency",
        ]])


show_terms_by_threshold(0.60, n=50)

threshold=0.6, min_margin=None, kept=3

[E] 2 candidates


,candidate_term,best_dimension_score,dimension_margin,best_seed_term,best_query_term,term_frequency,doc_frequency
0,전력 사용량,0.635120,0.339489,전력,전력사용량,26,10
1,에너지 사용량,0.627394,0.321419,전력,전력사용량,382,62



[S] 0 candidates


,candidate_term,best_dimension_score,dimension_margin,best_seed_term,best_query_term,term_frequency,doc_frequency



[G] 1 candidates


,candidate_term,best_dimension_score,dimension_margin,best_seed_term,best_query_term,term_frequency,doc_frequency
10034,ESG 위원회 사외 이사,0.712682,0.308162,사외이사,사외이사,25,25


## 9. seed별 top 후보 확인

In [10]:
top_rows = []

for seed_i, seed in seed_query_df.reset_index(drop=True).iterrows():
    sims = similarity_matrix[seed_i]
    top_idx = np.argsort(-sims)[:TOP_N_PER_SEED]
    for rank, cand_i in enumerate(top_idx, start=1):
        cand = candidate_df.iloc[cand_i]
        top_rows.append({
            "dimension": seed["dimension"],
            "seed_term": seed["seed_term"],
            "query_term": seed["query_term"],
            "rank": rank,
            "candidate_term": cand["candidate_term"],
            "similarity": float(sims[cand_i]),
            "term_frequency": cand["term_frequency"],
            "doc_frequency": cand["doc_frequency"],
        })

seed_top_candidate_df = pd.DataFrame(top_rows)
top_path = OUTPUT_DIR / "kiwi_embedding_top_candidates_by_seed.csv"
seed_top_candidate_df.to_csv(top_path, index=False, encoding="utf-8-sig")

print("saved:", top_path)
display(seed_top_candidate_df.head(50))

saved: /content/drive/MyDrive/UD_26/final/kiwi_embedding_seed_expansion/kiwi_embedding_top_candidates_by_seed.csv


,dimension,seed_term,query_term,rank,candidate_term,similarity,term_frequency,doc_frequency
0,E,탄소,탄소,1,탄소 발자국,0.534659,54,21
1,E,탄소,탄소,2,탄소 배출량,0.534569,113,42
2,E,탄소,탄소,3,탄소 배출,0.531883,166,59
3,E,탄소,탄소,4,탄소 녹색,0.512939,104,65
4,E,탄소,탄소,5,글로벌 탄소,0.511099,19,13
5,E,탄소,탄소,6,탄소 녹색 성장,0.501779,22,13
6,E,탄소,탄소,7,Carbon,0.501548,67,30
7,E,탄소,탄소,8,탄소 감축,0.499919,93,38
8,E,탄소,탄소,9,탄소 정책,0.488736,22,6
9,E,탄소,탄소,10,Carbon Capture,0.481942,21,7


## 10. 확장 사전 후보 CSV 저장

In [11]:
def regex_from_term(term: str) -> str:
    return re.escape(term)


def threshold_label(theta: float) -> str:
    return f"{theta:.2f}".replace(".", "_")


for theta in THRESHOLDS:
    kept = scored_candidate_df[scored_candidate_df["best_dimension_score"] >= theta].copy()
    kept = kept.rename(columns={"best_dimension": "dimension", "best_dimension_score": "similarity"})
    kept["dictionary_name"] = f"kiwi_embedding_theta_{threshold_label(theta)}"
    kept["threshold"] = theta
    kept["source"] = "kiwi_embedding_candidate"
    kept["seed_term"] = kept["best_seed_term"]
    kept["query_term"] = kept["best_query_term"]
    kept["pattern"] = kept["candidate_term"].map(regex_from_term)
    out_cols = [
        "dictionary_name", "threshold", "dimension", "source", "seed_term", "query_term",
        "candidate_term", "pattern", "similarity", "dimension_margin",
        "term_frequency", "doc_frequency",
    ]
    out_df = kept[out_cols].sort_values(
        ["dimension", "similarity", "dimension_margin", "candidate_term"],
        ascending=[True, False, False, True],
    )
    out_path = OUTPUT_DIR / f"kiwi_embedding_theta_{threshold_label(theta)}.csv"
    out_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("saved threshold dictionary candidates to:", OUTPUT_DIR)

saved threshold dictionary candidates to: /content/drive/MyDrive/UD_26/final/kiwi_embedding_seed_expansion


## 11. Threshold-level Spearman validation

This section converts each threshold-level expanded dictionary into firm-year TF-IDF features and compares them with KCGS ESG grades using Spearman rank correlation.

Notes:

- ESG grades are not used to generate candidates or choose thresholds; they are used only for validation.
- Because candidates are Kiwi noun/noun-phrase units, TF-IDF is computed on the same Kiwi noun/noun-phrase documents.
- Seed-only and expanded features are both calculated so preprocessing and expansion effects can be compared.


In [12]:
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer

GRADE_MAP = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5, "S": 6}
GRADE_BY_DIMENSION = {"E": "e_grade_num", "S": "s_grade_num", "G": "g_grade_num"}


def normalize_stock_code(series: pd.Series) -> pd.Series:
    return series.astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)


def token_key(term: str) -> str:
    return normalize_term(term).replace(" ", "_")


# Reuse the document-level Kiwi noun/noun-phrase terms created in section 4.
# This keeps candidate generation and correlation validation on the exact same preprocessing output.
if "kiwi_terms_by_doc" not in globals() or len(kiwi_terms_by_doc) != len(corpus_df):
    raise RuntimeError("Run section 4 first so kiwi_terms_by_doc is available for validation.")

kiwi_doc_df = corpus_df[["stock_code", "company_name", "fiscal_year", "esg_year", "total_word_count"]].copy()
kiwi_doc_df["kiwi_terms"] = kiwi_terms_by_doc
kiwi_doc_df["kiwi_document"] = [" ".join(token_key(term) for term in terms) for terms in kiwi_terms_by_doc]
kiwi_doc_df["kiwi_term_count"] = [len(terms) for terms in kiwi_terms_by_doc]

print("kiwi_doc rows:", len(kiwi_doc_df))
print("empty kiwi_document rows:", (kiwi_doc_df["kiwi_document"].str.len() == 0).sum())
display(kiwi_doc_df[["stock_code", "company_name", "fiscal_year", "kiwi_term_count", "kiwi_document"]].head())

kiwi_doc rows: 381
empty kiwi_document rows: 0


,stock_code,company_name,fiscal_year,kiwi_term_count,kiwi_document
0,000020,동화약품,2022,11248,II 사업 II_사업 II_사업 내용 사업 개요 일반 사항 지배 기업 사항_지배_기업 사항_지배 지배_기업 연결 실체 연결_실체 연결_실체 제공 재화 용역 근거 영업 부문 영업_부문 영업_부문 구분 부문 재무 정보 재무_정보 재무_정보 내부 관리 목적 내부_관리_목적 내부_관리 ...
1,000020,동화약품,2023,11303,II 사업 II_사업 II_사업 내용 사업 개요 일반 사항 지배 기업 사항_지배_기업 사항_지배 지배_기업 연결 실체 연결_실체 연결_실체 제공 재화 용역 근거 영업 부문 영업_부문 영업_부문 구분 부문 재무 정보 재무_정보 재무_정보 내부 관리 목적 내부_관리_목적 내부_관리 ...
2,000020,동화약품,2024,11469,II 사업 II_사업 II_사업 내용 사업 개요 일반 사항 지배 기업 사항_지배_기업 사항_지배 지배_기업 연결 실체 연결_실체 연결_실체 제공 재화 용역 근거 영업 부문 영업_부문 영업_부문 구분 부문 재무 정보 재무_정보 재무_정보 내부 관리 목적 내부_관리_목적 내부_관리 ...
3,000040,KR모터스,2022,6111,II 사업 II_사업 II_사업 내용 사업 개요 업계 현황 수출 주력 시장 현황_수출_주력_시장 현황_수출 수출_주력 주력_시장 유럽 경기 배기량 침체 지속 가운데 배기량 스쿠터 선호도 배기량_스쿠터_선호도 배기량_스쿠터 스쿠터_선호도 아시아 인도 중국 동남아시아 중심 시장 성장...
4,000040,KR모터스,2023,6728,II 사업 II_사업 II_사업 내용 사업 개요 업계 현황 수출 주력 시장 현황_수출_주력_시장 현황_수출 수출_주력 주력_시장 유럽 경기 배기량 침체 지속 가운데 배기량 스쿠터 선호도 배기량_스쿠터_선호도 배기량_스쿠터 스쿠터_선호도 아시아 인도 중국 동남아시아 중심 시장 성장...


In [13]:
# Fit TF-IDF on whitespace-separated Kiwi tokens. Multiword noun phrases are kept with underscores.
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=1,
    norm=None,
)

tfidf_matrix = vectorizer.fit_transform(kiwi_doc_df["kiwi_document"])
vocab = vectorizer.vocabulary_
print("tfidf_matrix:", tfidf_matrix.shape)
print("vocabulary size:", len(vocab))


def terms_present_in_vocab(terms: list[str]) -> list[str]:
    seen = []
    used = set()
    for term in terms:
        term = normalize_term(term)
        key = token_key(term)
        if term and key in vocab and term not in used:
            seen.append(term)
            used.add(term)
    return seen


def tfidf_sum_for_terms(terms: list[str]) -> np.ndarray:
    terms = terms_present_in_vocab(terms)
    if not terms:
        return np.zeros(tfidf_matrix.shape[0], dtype=float)
    cols = [vocab[token_key(term)] for term in terms]
    return np.asarray(tfidf_matrix[:, cols].sum(axis=1)).ravel()


def count_for_terms(terms: list[str]) -> np.ndarray:
    terms = terms_present_in_vocab(terms)
    if not terms:
        return np.zeros(len(kiwi_doc_df), dtype=int)
    term_set = set(terms)
    return np.asarray([sum(1 for term in doc_terms if term in term_set) for doc_terms in kiwi_doc_df["kiwi_terms"]], dtype=int)


tfidf_matrix: (381, 382517)
vocabulary size: 382517


In [14]:
# Create seed-only and threshold-specific expanded features.
score_df = kiwi_doc_df[["stock_code", "company_name", "fiscal_year", "esg_year", "total_word_count", "kiwi_term_count"]].copy()
feature_meta_rows = []

# Seed-only: sum seed query terms that are present in the Kiwi TF-IDF vocabulary.
for dimension in ["E", "S", "G"]:
    seed_terms_dim = seed_query_df.loc[seed_query_df["dimension"].eq(dimension), "query_term"].tolist()
    present_terms = terms_present_in_vocab(seed_terms_dim)
    feature_name = f"{dimension}_kiwi_seed_tfidf"
    count_name = f"{dimension}_kiwi_seed_count"
    score_df[feature_name] = tfidf_sum_for_terms(seed_terms_dim)
    score_df[count_name] = count_for_terms(seed_terms_dim)
    feature_meta_rows.append({
        "dictionary": "kiwi_seed_only",
        "threshold": np.nan,
        "dimension": dimension,
        "feature": feature_name,
        "term_count": len(present_terms),
        "terms_in_vocab": "|".join(present_terms),
    })

# Expanded: use scored_candidate_df candidates whose best_dimension_score is above each threshold.
for theta in THRESHOLDS:
    kept = scored_candidate_df[scored_candidate_df["best_dimension_score"] >= theta]
    for dimension in ["E", "S", "G"]:
        terms = kept.loc[kept["best_dimension"].eq(dimension), "candidate_term"].tolist()
        present_terms = terms_present_in_vocab(terms)
        theta_label = threshold_label(theta)
        feature_name = f"{dimension}_kiwi_expanded_{theta_label}_tfidf"
        count_name = f"{dimension}_kiwi_expanded_{theta_label}_count"
        score_df[feature_name] = tfidf_sum_for_terms(terms)
        score_df[count_name] = count_for_terms(terms)
        feature_meta_rows.append({
            "dictionary": "kiwi_embedding_expanded",
            "threshold": theta,
            "dimension": dimension,
            "feature": feature_name,
            "term_count": len(present_terms),
            "terms_in_vocab": "|".join(present_terms),
        })

feature_meta_df = pd.DataFrame(feature_meta_rows)
score_path = OUTPUT_DIR / "kiwi_embedding_threshold_scores.csv"
meta_path = OUTPUT_DIR / "kiwi_embedding_threshold_feature_terms.csv"
score_df.to_csv(score_path, index=False, encoding="utf-8-sig")
feature_meta_df.to_csv(meta_path, index=False, encoding="utf-8-sig")

print("saved:", score_path)
print("saved:", meta_path)
display(feature_meta_df[["dictionary", "threshold", "dimension", "term_count", "feature"]])


saved: /content/drive/MyDrive/UD_26/final/kiwi_embedding_seed_expansion/kiwi_embedding_threshold_scores.csv
saved: /content/drive/MyDrive/UD_26/final/kiwi_embedding_seed_expansion/kiwi_embedding_threshold_feature_terms.csv


,dictionary,threshold,dimension,term_count,feature
0,kiwi_seed_only,NaN,E,10,E_kiwi_seed_tfidf
1,kiwi_seed_only,NaN,S,12,S_kiwi_seed_tfidf
2,kiwi_seed_only,NaN,G,9,G_kiwi_seed_tfidf
3,kiwi_embedding_expanded,0.45,E,186,E_kiwi_expanded_0_45_tfidf
4,kiwi_embedding_expanded,0.45,S,36,S_kiwi_expanded_0_45_tfidf
5,kiwi_embedding_expanded,0.45,G,246,G_kiwi_expanded_0_45_tfidf
6,kiwi_embedding_expanded,0.50,E,75,E_kiwi_expanded_0_50_tfidf
7,kiwi_embedding_expanded,0.50,S,3,S_kiwi_expanded_0_50_tfidf
8,kiwi_embedding_expanded,0.50,G,98,G_kiwi_expanded_0_50_tfidf
9,kiwi_embedding_expanded,0.55,E,24,E_kiwi_expanded_0_55_tfidf


In [15]:
# Merge KCGS grades.
company_master = pd.read_csv(COMPANY_MASTER_PATH, dtype={"stock_code": "string"}, encoding="utf-8-sig")
company_master["stock_code"] = normalize_stock_code(company_master["stock_code"])
score_df["stock_code"] = normalize_stock_code(score_df["stock_code"])

for col in ["fiscal_year", "esg_year"]:
    company_master[col] = pd.to_numeric(company_master[col], errors="coerce").astype("Int64")
    score_df[col] = pd.to_numeric(score_df[col], errors="coerce").astype("Int64")

for col in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    company_master[f"{col}_num"] = company_master[col].map(GRADE_MAP)

grade_cols = [
    "stock_code", "fiscal_year", "esg_year", "industry",
    "esg_grade", "e_grade", "s_grade", "g_grade",
    "esg_grade_num", "e_grade_num", "s_grade_num", "g_grade_num",
]
grade_df = company_master[grade_cols].drop_duplicates(["stock_code", "fiscal_year", "esg_year"])
analysis_df = score_df.merge(grade_df, on=["stock_code", "fiscal_year", "esg_year"], how="left")

print("score rows:", len(score_df))
print("analysis rows:", len(analysis_df))
print("missing esg_grade_num:", analysis_df["esg_grade_num"].isna().sum())
display(analysis_df[["stock_code", "company_name", "fiscal_year", "esg_grade", "e_grade", "s_grade", "g_grade"]].head())

score rows: 381
analysis rows: 381
missing esg_grade_num: 0


,stock_code,company_name,fiscal_year,esg_grade,e_grade,s_grade,g_grade
0,000020,동화약품,2022,C,C,B,C
1,000020,동화약품,2023,C,B,B,C
2,000020,동화약품,2024,C,B,C,C
3,000040,KR모터스,2022,D,D,D,D
4,000040,KR모터스,2023,D,D,D,D


In [ ]:
# Spearman correlations: compare both TF-IDF and count scores.
def safe_spearman(x: pd.Series, y: pd.Series) -> tuple[float, float, int]:
    data = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(data) < 3 or data["x"].nunique() < 2 or data["y"].nunique() < 2:
        return np.nan, np.nan, len(data)
    rho, pvalue = spearmanr(data["x"], data["y"])
    return float(rho), float(pvalue), len(data)


def add_corr_row(rows: list[dict], dictionary: str, threshold, dimension: str, grade_col: str,
                 feature: str, score_type: str, term_count: int) -> None:
    rho, pvalue, n = safe_spearman(analysis_df[feature], analysis_df[grade_col])
    rows.append({
        "dictionary": dictionary,
        "threshold": threshold,
        "dimension": dimension,
        "score_type": score_type,
        "grade_col": grade_col,
        "feature": feature,
        "term_count": term_count,
        "spearman_rho": rho,
        "p_value": pvalue,
        "n": n,
    })


corr_rows = []

# Dimension-level correlations: E/S/G features use matching dimension grades.
for _, meta in feature_meta_df.iterrows():
    dimension = meta["dimension"]
    grade_col = GRADE_BY_DIMENSION[dimension]
    tfidf_feature = meta["feature"]
    count_feature = tfidf_feature.replace("_tfidf", "_count")
    add_corr_row(corr_rows, meta["dictionary"], meta["threshold"], dimension, grade_col,
                tfidf_feature, "tfidf", int(meta["term_count"]))
    add_corr_row(corr_rows, meta["dictionary"], meta["threshold"], dimension, grade_col,
                count_feature, "count", int(meta["term_count"]))

# Aggregate ESG score: sum E/S/G features and compare with total ESG grade.
for dictionary in ["kiwi_seed_only", "kiwi_embedding_expanded"]:
    thresholds = [np.nan] if dictionary == "kiwi_seed_only" else THRESHOLDS
    for theta in thresholds:
        if dictionary == "kiwi_seed_only":
            term_count = int(feature_meta_df.loc[feature_meta_df["dictionary"].eq(dictionary), "term_count"].sum())
            base_features = {
                "tfidf": [f"{dimension}_kiwi_seed_tfidf" for dimension in ["E", "S", "G"]],
                "count": [f"{dimension}_kiwi_seed_count" for dimension in ["E", "S", "G"]],
            }
            aggregate_features = {
                "tfidf": "ESG_kiwi_seed_tfidf",
                "count": "ESG_kiwi_seed_count",
            }
        else:
            theta_label = threshold_label(theta)
            term_count = int(feature_meta_df.loc[
                feature_meta_df["dictionary"].eq(dictionary) & feature_meta_df["threshold"].eq(theta),
                "term_count"
            ].sum())
            base_features = {
                "tfidf": [f"{dimension}_kiwi_expanded_{theta_label}_tfidf" for dimension in ["E", "S", "G"]],
                "count": [f"{dimension}_kiwi_expanded_{theta_label}_count" for dimension in ["E", "S", "G"]],
            }
            aggregate_features = {
                "tfidf": f"ESG_kiwi_expanded_{theta_label}_tfidf",
                "count": f"ESG_kiwi_expanded_{theta_label}_count",
            }

        for score_type in ["tfidf", "count"]:
            feature_name = aggregate_features[score_type]
            analysis_df[feature_name] = analysis_df[base_features[score_type]].sum(axis=1)
            add_corr_row(corr_rows, dictionary, theta, "ESG", "esg_grade_num", feature_name, score_type, term_count)

correlation_df = pd.DataFrame(corr_rows).sort_values(
    ["dimension", "score_type", "dictionary", "threshold"],
    na_position="first",
)
corr_path = OUTPUT_DIR / "kiwi_embedding_threshold_spearman.csv"
analysis_path = OUTPUT_DIR / "kiwi_embedding_threshold_analysis_panel.csv"
correlation_df.to_csv(corr_path, index=False, encoding="utf-8-sig")
analysis_df.to_csv(analysis_path, index=False, encoding="utf-8-sig")

print("saved:", corr_path)
print("saved:", analysis_path)
display(correlation_df)


saved: /content/drive/MyDrive/UD_26/final/kiwi_embedding_seed_expansion/kiwi_embedding_threshold_spearman.csv
saved: /content/drive/MyDrive/UD_26/final/kiwi_embedding_seed_expansion/kiwi_embedding_threshold_analysis_panel.csv


,dictionary,threshold,dimension,score_type,grade_col,feature,term_count,spearman_rho,p_value,n
7,kiwi_embedding_expanded,0.45,E,count,e_grade_num,E_kiwi_expanded_0_45_count,186,0.659004,8.112087e-49,381
13,kiwi_embedding_expanded,0.50,E,count,e_grade_num,E_kiwi_expanded_0_50_count,75,0.694052,4.802736e-56,381
19,kiwi_embedding_expanded,0.55,E,count,e_grade_num,E_kiwi_expanded_0_55_count,24,0.700216,2.002391e-57,381
25,kiwi_embedding_expanded,0.60,E,count,e_grade_num,E_kiwi_expanded_0_60_count,2,0.247299,1.022084e-06,381
31,kiwi_embedding_expanded,0.65,E,count,e_grade_num,E_kiwi_expanded_0_65_count,0,NaN,NaN,381
...,...,...,...,...,...,...,...,...,...,...
32,kiwi_embedding_expanded,0.65,S,tfidf,s_grade_num,S_kiwi_expanded_0_65_tfidf,0,NaN,NaN,381
38,kiwi_embedding_expanded,0.70,S,tfidf,s_grade_num,S_kiwi_expanded_0_70_tfidf,0,NaN,NaN,381
44,kiwi_embedding_expanded,0.75,S,tfidf,s_grade_num,S_kiwi_expanded_0_75_tfidf,0,NaN,NaN,381
50,kiwi_embedding_expanded,0.80,S,tfidf,s_grade_num,S_kiwi_expanded_0_80_tfidf,0,NaN,NaN,381


In [17]:
# Pivot tables for threshold-specific Spearman correlations.
for score_type in ["tfidf", "count"]:
    pivot_rho = correlation_df[
        correlation_df["dictionary"].eq("kiwi_embedding_expanded")
        & correlation_df["score_type"].eq(score_type)
    ].pivot_table(
        index="threshold",
        columns="dimension",
        values="spearman_rho",
        aggfunc="first",
    )
    print(f"Spearman rho by threshold - {score_type}")
    display(pivot_rho)

print("Seed-only baseline")
display(correlation_df[correlation_df["dictionary"].eq("kiwi_seed_only")])

print("Expanded dictionary term counts by threshold")
display(
    feature_meta_df[feature_meta_df["dictionary"].eq("kiwi_embedding_expanded")]
    .pivot(index="threshold", columns="dimension", values="term_count")
)


Spearman rho by threshold - tfidf


dimension,E,ESG,G,S
threshold,,,,
0.45,0.654992,0.715008,0.644645,0.505321
0.50,0.676846,0.713123,0.640550,0.131307
0.55,0.685048,0.710976,0.606260,NaN
0.60,0.248326,0.272491,0.132464,NaN
0.65,NaN,0.119960,0.132464,NaN
0.70,NaN,0.119960,0.132464,NaN


Spearman rho by threshold - count


dimension,E,ESG,G,S
threshold,,,,
0.45,0.659004,0.715997,0.625895,0.500815
0.50,0.694052,0.703739,0.616507,0.130298
0.55,0.700216,0.687612,0.585796,NaN
0.60,0.247299,0.271827,0.132464,NaN
0.65,NaN,0.119960,0.132464,NaN
0.70,NaN,0.119960,0.132464,NaN


Seed-only baseline


,dictionary,threshold,dimension,score_type,grade_col,feature,term_count,spearman_rho,p_value,n
1,kiwi_seed_only,NaN,E,count,e_grade_num,E_kiwi_seed_count,10,0.511493,8.685163e-27,381
0,kiwi_seed_only,NaN,E,tfidf,e_grade_num,E_kiwi_seed_tfidf,10,0.512143,7.313017e-27,381
55,kiwi_seed_only,NaN,ESG,count,esg_grade_num,ESG_kiwi_seed_count,31,0.691902,1.427727e-55,381
54,kiwi_seed_only,NaN,ESG,tfidf,esg_grade_num,ESG_kiwi_seed_tfidf,31,0.689479,4.819993e-55,381
5,kiwi_seed_only,NaN,G,count,g_grade_num,G_kiwi_seed_count,9,0.568412,5.504282e-34,381
4,kiwi_seed_only,NaN,G,tfidf,g_grade_num,G_kiwi_seed_tfidf,9,0.578072,2.375923e-35,381
3,kiwi_seed_only,NaN,S,count,s_grade_num,S_kiwi_seed_count,12,0.639163,3.858847e-45,381
2,kiwi_seed_only,NaN,S,tfidf,s_grade_num,S_kiwi_seed_tfidf,12,0.634205,2.908358e-44,381


Expanded dictionary term counts by threshold


dimension,E,G,S
threshold,,,
0.45,186,246,36
0.50,75,98,3
0.55,24,26,0
0.60,2,1,0
0.65,0,1,0
0.70,0,1,0
0.75,0,0,0
0.80,0,0,0


In [18]:
# Year-by-year Spearman correlations for robustness.
# Run this after correlation_df and analysis_df have been created.

year_corr_rows = []

for _, row in correlation_df.iterrows():
    feature = row["feature"]
    grade_col = row["grade_col"]

    if feature not in analysis_df.columns or grade_col not in analysis_df.columns:
        continue

    for fiscal_year, sub in analysis_df.groupby("fiscal_year"):
        rho, pvalue, n = safe_spearman(sub[feature], sub[grade_col])
        year_corr_rows.append({
            "dictionary": row["dictionary"],
            "threshold": row["threshold"],
            "dimension": row["dimension"],
            "score_type": row["score_type"],
            "fiscal_year": fiscal_year,
            "grade_col": grade_col,
            "feature": feature,
            "term_count": row["term_count"],
            "spearman_rho": rho,
            "p_value": pvalue,
            "n": n,
        })

year_correlation_df = pd.DataFrame(year_corr_rows).sort_values(
    ["dimension", "score_type", "dictionary", "threshold", "fiscal_year"],
    na_position="first",
)

year_corr_path = OUTPUT_DIR / "kiwi_embedding_threshold_spearman_by_year.csv"
year_correlation_df.to_csv(year_corr_path, index=False, encoding="utf-8-sig")

print("saved:", year_corr_path)
display(year_correlation_df)


# Compact pivot summaries for expanded dictionaries.
for score_type in ["tfidf", "count"]:
    year_pivot = year_correlation_df[
        year_correlation_df["dictionary"].eq("kiwi_embedding_expanded")
        & year_correlation_df["score_type"].eq(score_type)
    ].pivot_table(
        index=["threshold", "fiscal_year"],
        columns="dimension",
        values="spearman_rho",
        aggfunc="first",
    )

    print(f"Year-by-year Spearman rho - {score_type}")
    display(year_pivot)


# Seed-only yearly baseline.
print("Seed-only year-by-year baseline")
display(
    year_correlation_df[
        year_correlation_df["dictionary"].eq("kiwi_seed_only")
    ].sort_values(["dimension", "score_type", "fiscal_year"])
)


saved: /content/drive/MyDrive/UD_26/final/kiwi_embedding_seed_expansion/kiwi_embedding_threshold_spearman_by_year.csv


,dictionary,threshold,dimension,score_type,fiscal_year,grade_col,feature,term_count,spearman_rho,p_value,n
0,kiwi_embedding_expanded,0.45,E,count,2022,e_grade_num,E_kiwi_expanded_0_45_count,186,0.698534,6.869896e-20,127
1,kiwi_embedding_expanded,0.45,E,count,2023,e_grade_num,E_kiwi_expanded_0_45_count,186,0.677898,2.065179e-18,127
2,kiwi_embedding_expanded,0.45,E,count,2024,e_grade_num,E_kiwi_expanded_0_45_count,186,0.588203,3.557779e-13,127
3,kiwi_embedding_expanded,0.50,E,count,2022,e_grade_num,E_kiwi_expanded_0_50_count,75,0.716497,2.781147e-21,127
4,kiwi_embedding_expanded,0.50,E,count,2023,e_grade_num,E_kiwi_expanded_0_50_count,75,0.692246,1.997103e-19,127
...,...,...,...,...,...,...,...,...,...,...,...
211,kiwi_embedding_expanded,0.80,S,tfidf,2023,s_grade_num,S_kiwi_expanded_0_80_tfidf,0,NaN,NaN,127
212,kiwi_embedding_expanded,0.80,S,tfidf,2024,s_grade_num,S_kiwi_expanded_0_80_tfidf,0,NaN,NaN,127
213,kiwi_seed_only,NaN,S,tfidf,2022,s_grade_num,S_kiwi_seed_tfidf,12,0.729150,2.494413e-22,127
214,kiwi_seed_only,NaN,S,tfidf,2023,s_grade_num,S_kiwi_seed_tfidf,12,0.608578,3.203660e-14,127


Year-by-year Spearman rho - tfidf


dimension                     E       ESG         G         S
threshold fiscal_year                                        
0.45      2022         0.691843  0.739639  0.670679  0.586112
          2023         0.679225  0.710786  0.619916  0.490412
          2024         0.583943  0.690452  0.648485  0.424873
0.50      2022         0.710499  0.729843  0.662726  0.052574
          2023         0.680566  0.705287  0.620072  0.158015
          2024         0.631286  0.699476  0.645481  0.155072
0.55      2022         0.702894  0.740518  0.646710       NaN
          2023         0.682559  0.693172  0.591492       NaN
          2024         0.659144  0.694434  0.586301       NaN
0.60      2022         0.280470  0.265561  0.069746       NaN
          2023         0.287802  0.289922  0.159343       NaN
          2024         0.176401  0.261911  0.164002       NaN
0.65      2022              NaN  0.073028  0.069746       NaN
          2023              NaN  0.120916  0.159343       NaN
          2024              NaN  0.159098  0.164002       NaN
0.70      2022              NaN  0.073028  0.069746       NaN
          2023              NaN  0.120916  0.159343       NaN
          2024              NaN  0.159098  0.164002       NaN

Year-by-year Spearman rho - count


dimension                     E       ESG         G         S
threshold fiscal_year                                        
0.45      2022         0.698534  0.740949  0.662235  0.581715
          2023         0.677898  0.704174  0.599992  0.486925
          2024         0.588203  0.693026  0.619735  0.421335
0.50      2022         0.716497  0.728611  0.645588  0.052093
          2023         0.692246  0.685587  0.598304  0.157065
          2024         0.660031  0.694300  0.607400  0.152856
0.55      2022         0.706154  0.720474  0.626775       NaN
          2023         0.697662  0.667895  0.567128       NaN
          2024         0.686293  0.670027  0.569827       NaN
0.60      2022         0.279535  0.264494  0.069746       NaN
          2023         0.286509  0.289708  0.159343       NaN
          2024         0.176049  0.261277  0.164002       NaN
0.65      2022              NaN  0.073028  0.069746       NaN
          2023              NaN  0.120916  0.159343       NaN
          2024              NaN  0.159098  0.164002       NaN
0.70      2022              NaN  0.073028  0.069746       NaN
          2023              NaN  0.120916  0.159343       NaN
          2024              NaN  0.159098  0.164002       NaN

Seed-only year-by-year baseline


,dictionary,threshold,dimension,score_type,fiscal_year,grade_col,feature,term_count,spearman_rho,p_value,n
24,kiwi_seed_only,NaN,E,count,2022,e_grade_num,E_kiwi_seed_count,10,0.542982,4.249859e-11,127
25,kiwi_seed_only,NaN,E,count,2023,e_grade_num,E_kiwi_seed_count,10,0.535100,9.110783e-11,127
26,kiwi_seed_only,NaN,E,count,2024,e_grade_num,E_kiwi_seed_count,10,0.452377,9.327753e-08,127
51,kiwi_seed_only,NaN,E,tfidf,2022,e_grade_num,E_kiwi_seed_tfidf,10,0.539154,6.170344e-11,127
52,kiwi_seed_only,NaN,E,tfidf,2023,e_grade_num,E_kiwi_seed_tfidf,10,0.538282,6.712767e-11,127
53,kiwi_seed_only,NaN,E,tfidf,2024,e_grade_num,E_kiwi_seed_tfidf,10,0.450028,1.106921e-07,127
78,kiwi_seed_only,NaN,ESG,count,2022,esg_grade_num,ESG_kiwi_seed_count,31,0.708334,1.230854e-20,127
79,kiwi_seed_only,NaN,ESG,count,2023,esg_grade_num,ESG_kiwi_seed_count,31,0.702254,3.606228e-20,127
80,kiwi_seed_only,NaN,ESG,count,2024,esg_grade_num,ESG_kiwi_seed_count,31,0.666503,1.204388e-17,127
105,kiwi_seed_only,NaN,ESG,tfidf,2022,esg_grade_num,ESG_kiwi_seed_tfidf,31,0.708567,1.180587e-20,127
